# 4 · CD4/CD8 deconvolution — is the descriptor compositional?

Notebook 02 showed repertoires *classify* CD4 vs CD8. Here the sharper question: can we read the
**fraction** of CD4/CD8 in an unsorted (bulk) repertoire? This is the vaccine-relevant question —
estimating immune composition without cell sorting.

Four questions, building up:
- **Q1** — does a synthetic mixture's descriptor move smoothly with the mixing fraction?
- **Q2** — can we estimate the fraction *cross-patient* (reference poles from other patients)?
- **Q3** — does occupancy deconvolve better than mean+cov (mixing is linear on histograms)?
- **Q4** — what is the error in the physiological blood range (the number that matters)?

**Honest scope.** The "bulk" is a **synthetic in-silico mix** of FACS-sorted CD4/CD8 clouds — labels are
cytometry ground truth, but the mixture is not real unsorted bulk. This is a proof of concept; the next
step is real bulk with cytometry-measured fractions.

## Setup
Load raw clonotype embeddings (not just descriptors) — we need the individual clonotypes to build mixtures.

In [ ]:
import sys, re, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

REPERTOIRE_DIR = '../scripts/repertoire'
CLOUDS_DIR     = '../data/clouds_embedded_TRB'
LANDMARKS      = '../data/landmarks_beta.npz'
sys.path.insert(0, REPERTOIRE_DIR)
import rep_data, rep_descriptors as rd
EMB = rep_data.EMB_COLS  # e0..e127

def parse(s):
    pid = re.match(r'(MS\d+|T1D\d+|HD_[A-Za-z]+)', s).group(1)
    cell = 'CD4' if 'CD4' in s else 'CD8'
    return pid, cell

def load_raw(path):
    """Clonotype embeddings Z (L2-normalized) and abundance weights w."""
    df = pd.read_parquet(path, columns=EMB + ['w_log'])
    Z = df[EMB].to_numpy(np.float32); Z /= (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-8)
    w = df['w_log'].to_numpy(np.float32)
    return Z, w

files = rep_data.cloud_files(CLOUDS_DIR)
bykey = {}
for s in files:
    pid, cell = parse(s); bykey.setdefault(pid, {})[cell] = files[s]
paired = [p for p, d in bykey.items() if 'CD4' in d and 'CD8' in d]
print(f'{len(paired)} patients with both CD4 and CD8')

## Q1 · Does the descriptor track the mixing fraction?

Take one patient's CD4 and CD8 clonotypes, build synthetic mixtures at known CD8 fractions (0 → 1),
and measure where each mixture's descriptor sits on the CD4→CD8 axis (cosine to each pole).

**Insight:** the position increases monotonically with CD8 fraction (Spearman ρ ≈ 1) → the descriptor
encodes composition → deconvolution is feasible.

In [ ]:
patient = paired[0]
print(f'Using patient: {patient}')
Z4, w4 = load_raw(bykey[patient]['CD4'])
Z8, w8 = load_raw(bykey[patient]['CD8'])

# the two poles (pure-class descriptors)
d_cd4 = rd.mean_cov_weighted_np(Z4, w4 / w4.sum())
d_cd8 = rd.mean_cov_weighted_np(Z8, w8 / w8.sum())
d4n = d_cd4 / np.linalg.norm(d_cd4)
d8n = d_cd8 / np.linalg.norm(d_cd8)

# synthetic mixtures at known CD8 fractions
fracs = np.linspace(0, 1, 11)
Zmix = np.vstack([Z4, Z8])
proj = []
for f in fracs:
    wmix = np.concatenate([(1 - f) * w4 / w4.sum(), f * w8 / w8.sum()]); wmix /= wmix.sum()
    dm = rd.mean_cov_weighted_np(Zmix, wmix); dm /= (np.linalg.norm(dm) + 1e-8)
    proj.append(dm @ d8n - dm @ d4n)   # >0 = closer to CD8

rho, p = spearmanr(fracs, proj)
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(fracs, proj, marker='o', markersize=8, lw=2.5, color='#065A82')
ax.axhline(0, color='gray', ls=':', lw=1, alpha=0.6)
ax.set_xlabel('CD8 fraction in synthetic bulk mixture', fontsize=12, fontweight='bold')
ax.set_ylabel('Descriptor position on CD4\u2192CD8 axis\n(cos to CD8 \u2212 cos to CD4)', fontsize=12, fontweight='bold')
ax.set_title(f'Descriptor tracks mixing fraction  (patient {patient}, Spearman \u03c1={rho:.3f})', fontsize=12)
ax.grid(True, alpha=0.25)
plt.tight_layout(); plt.show()
print(f'Spearman rho = {rho:.3f} (p={p:.2e}) -> monotonic -> deconvolution feasible')

## Q2 · Cross-patient deconvolution (leave-one-patient-out)

The real test: estimate a bulk's CD8 fraction using reference poles built from **other** patients
(leave-one-patient-out). If it generalizes, the poles are not patient-specific.

Method: build global CD4/CD8 poles from training patients; for the held-out patient, make synthetic
bulks of known fraction, project onto the axis, and linearly invert to a predicted fraction.

**Insight:** predictions track the truth (MAE ≈ 0.147) with reference poles from strangers → the
CD4/CD8 axis is shared across patients.

In [ ]:
def pole_from_patients(patient_list, cell):
    ds = []
    for p in patient_list:
        if cell in bykey[p]:
            Z, w = load_raw(bykey[p][cell])
            d = rd.mean_cov_weighted_np(Z, w / w.sum())
            ds.append(d / (np.linalg.norm(d) + 1e-8))
    return np.mean(ds, axis=0)

true_fracs = np.linspace(0, 1, 11)
all_true, all_pred = [], []
for test_p in paired:
    train_p = [p for p in paired if p != test_p]
    d4 = pole_from_patients(train_p, 'CD4'); d4 /= np.linalg.norm(d4)
    d8 = pole_from_patients(train_p, 'CD8'); d8 /= np.linalg.norm(d8)
    Z4, w4 = load_raw(bykey[test_p]['CD4']); Z8, w8 = load_raw(bykey[test_p]['CD8'])
    Zmix = np.vstack([Z4, Z8])
    pos4 = (d4 @ d8) - (d4 @ d4)   # pure-CD4 calibration point
    pos8 = (d8 @ d8) - (d8 @ d4)   # pure-CD8 calibration point
    for f in true_fracs:
        wmix = np.concatenate([(1 - f) * w4 / w4.sum(), f * w8 / w8.sum()]); wmix /= wmix.sum()
        dm = rd.mean_cov_weighted_np(Zmix, wmix); dm /= np.linalg.norm(dm)
        pos = (dm @ d8) - (dm @ d4)
        pred = (pos - pos4) / (pos8 - pos4 + 1e-8)   # invert to fraction
        all_true.append(f); all_pred.append(pred)
all_true = np.array(all_true); all_pred = np.array(all_pred)
mae = np.mean(np.abs(all_pred - all_true)); rho, _ = spearmanr(all_true, all_pred)
print(f'Cross-patient (leave-one-patient-out): MAE = {mae:.3f} | Spearman = {rho:.3f}')

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(all_true, all_pred, alpha=0.35, s=40, color='#065A82', edgecolor='none')
for f in true_fracs:
    m = all_true == f
    ax.errorbar(f, all_pred[m].mean(), yerr=all_pred[m].std(), fmt='s', color='#C0392B', markersize=6, capsize=3, zorder=5)
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.7, label='perfect (y = x)')
ax.set_xlabel('True CD8 fraction in bulk', fontsize=12, fontweight='bold')
ax.set_ylabel('Estimated CD8 fraction\n(poles from OTHER patients)', fontsize=12, fontweight='bold')
ax.set_title(f'Cross-patient CD4/CD8 deconvolution  (MAE={mae:.3f}, n={len(paired)})', fontsize=12)
ax.legend(loc='upper left', fontsize=10); ax.grid(True, alpha=0.25)
ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.1, 1.15)
plt.tight_layout(); plt.show()

## Q3 · Does occupancy deconvolve better than mean+cov?

Mixing repertoires is a **linear** operation on occupancy histograms (a mixture's histogram is the
weighted average of the class histograms). That additivity should make occupancy well-suited to
deconvolution. Compare both descriptors head-to-head.

**Insight:** occupancy gives lower error (MAE ≈ 0.132 vs 0.147), consistent with its additive nature.

In [ ]:
protos = np.load(LANDMARKS)['centroids']

def desc_meancov(Z, w):
    d = rd.mean_cov_weighted_np(Z, w / w.sum()); return d / (np.linalg.norm(d) + 1e-8)
def desc_occ(Z, w):
    d = rd.weighted_occupancy(Z, w / w.sum(), protos, tau=0.1); return d / (np.linalg.norm(d) + 1e-8)
def pole(patient_list, cell, descfn):
    ds = [descfn(*load_raw(bykey[p][cell])) for p in patient_list if cell in bykey[p]]
    m = np.mean(ds, axis=0); return m / (np.linalg.norm(m) + 1e-8)

def deconvolve(descfn):
    T, P = [], []
    for test_p in paired:
        train_p = [p for p in paired if p != test_p]
        d4 = pole(train_p, 'CD4', descfn); d8 = pole(train_p, 'CD8', descfn)
        Z4, w4 = load_raw(bykey[test_p]['CD4']); Z8, w8 = load_raw(bykey[test_p]['CD8'])
        Zmix = np.vstack([Z4, Z8])
        pos4 = (d4 @ d8) - (d4 @ d4); pos8 = (d8 @ d8) - (d8 @ d4)
        for f in true_fracs:
            wmix = np.concatenate([(1 - f) * w4 / w4.sum(), f * w8 / w8.sum()]); wmix /= wmix.sum()
            dm = descfn(Zmix, wmix)
            pos = (dm @ d8) - (dm @ d4)
            T.append(f); P.append((pos - pos4) / (pos8 - pos4 + 1e-8))
    return np.array(T), np.array(P)

results = {}
for name, fn in [('mean+cov', desc_meancov), ('occupancy', desc_occ)]:
    T, P = deconvolve(fn); mae = np.mean(np.abs(P - T)); rho, _ = spearmanr(T, P)
    results[name] = (T, P, mae, rho)
    print(f'{name:>10}:  MAE = {mae:.3f}  |  Spearman = {rho:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharex=True, sharey=True)
colors = {'mean+cov':'#065A82', 'occupancy':'#C0392B'}
for ax, name in zip(axes, ['mean+cov', 'occupancy']):
    T, P, mae, rho = results[name]
    ax.scatter(T, P, alpha=0.3, s=35, color=colors[name], edgecolor='none')
    for f in true_fracs:
        m = T == f
        ax.errorbar(f, P[m].mean(), yerr=P[m].std(), fmt='s', color='black', markersize=5, capsize=3, zorder=5)
    ax.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.7, label='perfect (y = x)')
    ax.set_xlabel('True CD8 fraction in bulk', fontsize=12, fontweight='bold')
    ax.set_title(f'{name}\nMAE = {mae:.3f}  |  Spearman = {rho:.3f}', fontsize=12)
    ax.legend(loc='upper left', fontsize=9); ax.grid(True, alpha=0.25)
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.1, 1.15)
axes[0].set_ylabel('Estimated CD8 fraction\n(poles from OTHER patients)', fontsize=12, fontweight='bold')
fig.suptitle('Deconvolution: mean+cov vs occupancy (leave-one-patient-out)', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()
winner = min(results, key=lambda k: results[k][2])
print(f'\nLower error: {winner} (MAE {results[winner][2]:.3f})')

## Q4 · What is the error in the physiological range?

The number that matters for real samples. In blood, CD4:CD8 ≈ 2:1 to 3:1, i.e. CD8 fraction ≈ 0.25–0.40.
Using occupancy (the winner), what is the deconvolution error *inside that range*?

**Insight:** error stays flat (~0.13) across the physiological range — the estimate is as reliable there
as anywhere on the 0–1 axis, which is what a real vaccine application would need.

In [ ]:
T, P = deconvolve(desc_occ)   # occupancy = winner
phys_lo, phys_hi = 0.25, 0.40
mask_phys = (T >= phys_lo - 1e-6) & (T <= phys_hi + 1e-6)
mae_all = np.mean(np.abs(P - T))
mae_phys = np.mean(np.abs(P[mask_phys] - T[mask_phys])) if mask_phys.sum() else float('nan')
print(f'MAE over full 0-1 range : {mae_all:.3f}')
print(f'MAE in physiological range ({phys_lo}-{phys_hi}): {mae_phys:.3f} (n={mask_phys.sum()})')

mae_by_frac = [np.mean(np.abs(P[T == f] - f)) for f in true_fracs]
fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.plot(true_fracs, mae_by_frac, marker='o', markersize=8, lw=2.5, color='#C0392B')
ax.axvspan(phys_lo, phys_hi, color='#2C7FB8', alpha=0.15, label='physiological blood range\n(CD4:CD8 \u2248 2:1)')
ax.set_xlabel('True CD8 fraction in bulk', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean absolute error of estimate', fontsize=12, fontweight='bold')
ax.set_title('Where is deconvolution most reliable? (occupancy)', fontsize=12)
ax.grid(True, alpha=0.25); ax.legend(loc='upper center', fontsize=10)
ax.set_ylim(0, max(mae_by_frac) * 1.15)
plt.tight_layout(); plt.show()

## Summary

| Question | Result |
|---|---|
| Q1 · Descriptor tracks fraction? | **Yes** — monotonic, Spearman ρ ≈ 1 |
| Q2 · Cross-patient estimate? | **Yes** — MAE ≈ 0.147 (leave-one-patient-out) |
| Q3 · Occupancy better? | **Yes** — MAE ≈ 0.132 (additivity helps) |
| Q4 · Physiological range error? | ~0.13, flat across CD8 ≈ 0.25–0.40 |

**Takeaway.** The repertoire descriptor is compositional: the CD4/CD8 fraction of a mixture is recoverable cross-patient, with occupancy slightly ahead thanks to its additive structure.

Scope, precisely. The source data are already FACS-sorted into separate CD4 and CD8 populations. The "bulk" here is therefore synthetic: the two sorted populations are mixed in silico at known fractions, and the method recovers those known fractions. This is a controlled proof of concept — it shows the descriptor carries fraction information — but it is not real unsorted bulk. The next step is to test on genuinely unsorted repertoires with the CD4/CD8 fraction measured independently (e.g. by cytometry).